# ECAPA-TDNN + SNN FiLM conditioning (VoxCeleb2 → VoxCeleb1)

Conditions ECAPA-TDNN on the triplet-STDP SNN fingerprint from
`training/ecapa_film_snn/prepare_fingerprints.ipynb` via **FiLM**: a small MLP turns the
fingerprint into per-channel scale/shift `(γ, β)`, applied to ECAPA's post-MFA feature map
(`layer4`'s output, 1536 channels) right before attentive statistics pooling — `h' = γ·h + β`.

**Ablation**: each fingerprint component (`in_weights`, `hid_weights`, `input_activity`,
`hidden_activity`) is independently toggleable via `FINGERPRINT_PARTS` in the CONFIG cell. A
disabled component's encoder branch is skipped and replaced with a constant near-zero vector
(`FILM_DISABLED_EPS`, not an exact zero) — the FiLM generator's architecture and parameter
count stay **identical across every ablation config**, so a change in EER is attributable only
to the missing signal, not to a smaller model. `USE_FILM=False` skips the FiLM module
entirely (an architecture-level ablation on top of the component-level one).

**Goal**: match or slightly beat the `USE_FILM=False` result — evidence the SNN identity
signal adds real value, not just redundant rate/energy info ECAPA's own mel front-end already
captures.

**No augmentation**: unlike the standard ECAPA recipe, this run applies neither SpecAugment
nor MUSAN/RIR. The FiLM fingerprint is computed once from the clean, continuous 2s clip
(`prepare_fingerprints.ipynb`); augmenting the waveform or mel features ECAPA actually trains
on would decouple them from what the fingerprint describes. This also matches how the SNN
itself was trained — continuous audio, no augmentation. Expect somewhat worse absolute EER
than the reference recipe (which leans on augmentation); this is an internal ablation, not a
benchmark run.

## Data pipeline — fingerprints drive enumeration

Every training/eval clip here is one `prepare_fingerprints.ipynb` already fingerprinted,
located by `(person_id, session_id, file_name)` — not an independent corpus walk. This
guarantees every clip has exactly the right fingerprint, by construction: **prerequisite** —
run `prepare_fingerprints.ipynb` over every train (vox2) and eval (vox1) shard first, and
upload the resulting npz collection as a Kaggle dataset; point `FINGERPRINT_TRAIN_GLOB` /
`FINGERPRINT_TEST_GLOB` at it below.

Both train and eval audio are decoded to only their **first 2 seconds** (`CLIP_SEC`), matching
the SNN's own `CLIP_MS=2000` window exactly — memory-efficient for the full-corpus decode
cache, and keeps every ablation config comparable on identical audio. ffmpeg is told to stop
decoding at `CLIP_SEC` (`-t`) rather than decoding each full file and truncating afterward, so
the decode cache only pays for the audio it keeps. One consequence: with the stored clip length
equal to the training crop length, random-crop-position augmentation degenerates to a no-op —
and since there is no other augmentation here either, training sees the exact same 2s window
every epoch.

| | |
|---|---|
| model | ECAPA-TDNN, `C=1024`, 192-d embedding (~15.4M params) + FiLM generator (~0.6M params) |
| FiLM point | post-MFA feature map (`layer4` output, 1536 channels), right before attentive stats pooling |
| FiLM input | `in_weights` (2,23,128), `hid_weights` (2,23,128), `input_activity` (128,), `hidden_activity` (128,) |
| features | 80-mel, 16 kHz, n_fft 512 / win 400 / hop 160, 20–7600 Hz, pre-emphasis 0.97 |
| audio window | first 2s of every clip, train AND eval, no augmentation |
| loss | AAMSoftmax, m=0.2, s=30 |
| optim | Adam, lr 1e-3, wd 2e-5, lr × 0.97 per epoch, 80 epochs |
| batch | 128 |
| ablation | edit `USE_FILM` / `FINGERPRINT_PARTS` in CONFIG and rerun — `TAG` encodes the active config so checkpoints from different ablation runs never collide |

**Eval protocol unchanged**: session-free all-pairs cosine retrieval (EER / Rank-1 / Rank-5 /
mAP), same as `ann_backend.ipynb`.

In [ ]:
# ── Environment check (Kaggle ships torch, torchaudio, soundfile, scipy, ffmpeg) ──
import shutil, sys
import torch, torchaudio
print("torch     ", torch.__version__)
print("torchaudio", torchaudio.__version__)
print("cuda      ", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
assert shutil.which("ffmpeg"), "ffmpeg not on PATH — needed to decode VoxCeleb2 .m4a"
print("ffmpeg    ", shutil.which("ffmpeg"))

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  CONFIG  —  the only cell you normally edit
# ══════════════════════════════════════════════════════════════════════════════
import os, glob, math, time, random
import numpy as np
import torch

SEED = 1234
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

# ---- Fingerprints (drive enumeration — every clip trained/evaluated on is one
#      that prepare_fingerprints.ipynb already fingerprinted) ------------------
# Point these at wherever you upload prepare_fingerprints.ipynb's output collection.
FINGERPRINT_TRAIN_GLOB = "/kaggle/input/datasets/qphulong/vox1and2-fingerprints-tripletstdp/vox2_*_fingerprints.npz"
FINGERPRINT_TEST_GLOB  = "/kaggle/input/datasets/qphulong/vox1and2-fingerprints-tripletstdp/vox1_*_fingerprints.npz"

# ---- Raw audio roots (used only to resolve each fingerprinted clip's wav path) -
TRAIN_GLOB = "/kaggle/input/datasets/qphulong/vox2-voices-200person-shard*"
TEST_GLOB  = "/kaggle/input/datasets/qphulong/vox1-voices"   # contains dev_NN/ subfolders

# ---- Audio ----------------------------------------------------------------------
# No augmentation (no SpecAugment, no MUSAN/RIR): the FiLM fingerprint is computed
# once from the clean, continuous 2s clip, so ECAPA trains on that same clean clip
# rather than a corrupted/masked version of it — and this matches how the SNN
# itself was trained (continuous audio, no augmentation).
SR            = 16000
CLIP_SEC      = 2.0                # first CLIP_SEC of every clip — train AND eval —
                                    # matching the fingerprint's own CLIP_MS window
                                    # exactly (memory-efficient, keeps every
                                    # ablation config comparable on identical audio)
CROP_SEC      = CLIP_SEC           # kept as a separate name for clarity at call
                                    # sites; equal to CLIP_SEC by design
CROP_SAMPLES  = int(CROP_SEC * SR)
TRAIN_CAP_SEC = CLIP_SEC
TEST_CAP_SEC  = CLIP_SEC

# ---- Decode cache (scratch disk, NOT /kaggle/working which is output-quota'd) --
CACHE_DIR = "/kaggle/temp/ecapa_cache"

# ---- Model ------------------------------------------------------------------------
C          = 1024        # reference default; 512 (~7.0M params) also standard
EMBED_DIM  = 192

# ---- FiLM conditioning (SNN fingerprint -> gamma/beta on the post-MFA map) -------
USE_FILM = True     # False: FiLM module is skipped entirely (architecture-level
                     # ablation, not just zeroed inputs) — same data pipeline either way.
FINGERPRINT_PARTS = dict(
    in_weights=True,        # (2,23,128) in->hid STDP weights
    hid_weights=True,       # (2,23,128) hid->hid STDP weights
    input_activity=True,    # (128,) input firing rate
    hidden_activity=True,   # (128,) hidden firing rate
)
FILM_BRANCH_DIM   = 64      # per-component encoder output dim
FILM_HIDDEN       = 128     # shared trunk hidden dim
FILM_DISABLED_EPS = 1e-6    # a disabled branch contributes this constant, not an
                             # exact zero — keeps it out of a degenerate all-zero
                             # LayerNorm regime while still carrying ~no information
FILM_GAMMA_RANGE  = 1.0     # gamma is bounded to [1-FILM_GAMMA_RANGE, 1+FILM_GAMMA_RANGE]
                             # via tanh. Without this, gamma = 1 + Linear(...) is
                             # unbounded: a few bad gradient steps can push it large,
                             # multiplicatively blowing up the 1536-channel feature map
                             # under fp16 autocast -> NaN loss for the rest of training.
                             # This was a real failure mode here even with clean,
                             # NaN/Inf-free fingerprint data — a training-dynamics bug,
                             # not a data bug. tanh(0)=0 at init, so gamma=1 (identity)
                             # is preserved exactly regardless of this bound.
FILM_BETA_SCALE   = 1.0     # beta is bounded to [-FILM_BETA_SCALE, FILM_BETA_SCALE]
                             # via tanh, same rationale as FILM_GAMMA_RANGE.

# ---- Optim / schedule (reference recipe) ------------------------------------------
EPOCHS       = 80
BATCH_SIZE   = 128       # reference uses 400; 128 is sized for a 16GB Kaggle GPU at C=1024
LR           = 1e-3
LR_DECAY     = 0.97      # multiplied per epoch
WEIGHT_DECAY = 2e-5
AAM_M        = 0.2
AAM_S        = 30
GRAD_CLIP    = 5.0

# ---- Eval ---------------------------------------------------------------------------
EVAL_EVERY   = 1
EVAL_BATCH   = 32        # length-sorted buckets

# ---- Checkpointing --------------------------------------------------------------------
RESUME    = True
CKPT_DIR  = "/kaggle/working"
_active_parts = "".join(k[0] for k, v in FINGERPRINT_PARTS.items() if v)
_film_tag = f"film_{_active_parts or 'none'}" if USE_FILM else "nofilm"
TAG       = f"ecapa_C{C}_{_film_tag}"
LAST_CKPT = os.path.join(CKPT_DIR, f"last_{TAG}.pt")
BEST_CKPT = os.path.join(CKPT_DIR, f"best_{TAG}.pt")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
os.makedirs(CACHE_DIR, exist_ok=True)
os.makedirs(CKPT_DIR, exist_ok=True)
print(f"device={device}  TAG={TAG}  C={C}  batch={BATCH_SIZE}  epochs={EPOCHS}")
print(f"USE_FILM={USE_FILM}  parts={FINGERPRINT_PARTS if USE_FILM else '-'}")
print(f"cache  ={CACHE_DIR}")

In [ ]:
# ── Merge fingerprint shards into one memmapped cache ───────────────────────────
# Memory-efficient: only ONE shard's arrays are ever resident in RAM at a time
# during the merge; the merged output itself is written straight to a memmapped
# .npy so training never loads the full fingerprint corpus into RAM either.
def build_fingerprint_cache(glob_pattern, name):
    """Merge every shard npz matching glob_pattern into
    {CACHE_DIR}/{name}_fp_*.npy (memmap-able) + {name}_fp_meta.npz
    (person_ids/session_ids/file_names). Idempotent: re-running with the same
    shard set hits the cache."""
    shard_paths = sorted(glob.glob(glob_pattern))
    assert shard_paths, f"[{name} fp] nothing matches {glob_pattern!r}"

    prefix    = os.path.join(CACHE_DIR, f"{name}_fp")
    meta_path = f"{prefix}_meta.npz"
    if os.path.exists(meta_path):
        m = np.load(meta_path, allow_pickle=True)
        if list(m["shard_paths"]) == shard_paths:
            print(f"[{name} fp] cache hit: {int(m['n_rows'])} rows from "
                  f"{len(shard_paths)} shards")
            return prefix, meta_path
        print(f"[{name} fp] shard set changed — rebuilding")

    # Pass 1: row counts only (mmap, no data read) so the output arrays can be
    # allocated once instead of grown/reallocated.
    counts = []
    for p in shard_paths:
        z = np.load(p, mmap_mode="r")
        counts.append(z["in_weights"].shape[0])
    n_rows = sum(counts)
    assert n_rows > 0, f"[{name} fp] shards matched but contained 0 rows"

    iw = np.lib.format.open_memmap(f"{prefix}_in_weights.npy",      mode="w+",
                                   dtype=np.float16, shape=(n_rows, 2, 23, 128))
    hw = np.lib.format.open_memmap(f"{prefix}_hid_weights.npy",     mode="w+",
                                   dtype=np.float16, shape=(n_rows, 2, 23, 128))
    ia = np.lib.format.open_memmap(f"{prefix}_input_activity.npy",  mode="w+",
                                   dtype=np.float16, shape=(n_rows, 128))
    ha = np.lib.format.open_memmap(f"{prefix}_hidden_activity.npy", mode="w+",
                                   dtype=np.float16, shape=(n_rows, 128))

    person_ids, session_ids, file_names = [], [], []
    pos = 0
    for p, n in zip(shard_paths, counts):
        z = np.load(p, allow_pickle=True)   # one shard's arrays in RAM at a time
        iw[pos:pos + n] = z["in_weights"]
        hw[pos:pos + n] = z["hid_weights"]
        ia[pos:pos + n] = z["input_activity"]
        ha[pos:pos + n] = z["hidden_activity"]
        person_ids.extend(z["person_ids"].tolist())
        session_ids.extend(z["session_ids"].tolist())
        file_names.extend(z["file_names"].tolist())
        pos += n
        del z
    iw.flush(); hw.flush(); ia.flush(); ha.flush()

    np.savez(meta_path, shard_paths=np.array(shard_paths), n_rows=n_rows,
             person_ids=np.array(person_ids), session_ids=np.array(session_ids),
             file_names=np.array(file_names))
    print(f"[{name} fp] merged {n_rows} rows from {len(shard_paths)} shards "
          f"-> {prefix}_*.npy")
    return prefix, meta_path


class FingerprintBank:
    """Read-only, memmapped view over one merged fingerprint cache."""
    def __init__(self, prefix, meta_path):
        m = np.load(meta_path, allow_pickle=True)
        self.person_ids  = m["person_ids"].astype(str)
        self.session_ids = m["session_ids"].astype(str)
        self.file_names  = m["file_names"].astype(str)
        self.in_weights      = np.load(f"{prefix}_in_weights.npy",      mmap_mode="r")
        self.hid_weights     = np.load(f"{prefix}_hid_weights.npy",     mmap_mode="r")
        self.input_activity  = np.load(f"{prefix}_input_activity.npy",  mmap_mode="r")
        self.hidden_activity = np.load(f"{prefix}_hidden_activity.npy", mmap_mode="r")
        self.n = len(self.person_ids)

    def batch(self, rows, device):
        rows = np.asarray(rows)
        def _get(arr):
            t = torch.from_numpy(np.asarray(arr[rows])).to(device).float()
            # A fingerprint value that is NaN/Inf (e.g. an STDP weight that
            # overflowed float16's ~65504 max during storage) would otherwise
            # poison FingerprintFiLM's zero-initialised gamma/beta: 0 * NaN is
            # NaN, not 0, so the "starts as identity" guarantee silently fails
            # and the whole batch's loss goes NaN from step 1. Zero is the
            # same "no information" value FILM_DISABLED_EPS approximates.
            return torch.nan_to_num(t, nan=0.0, posinf=0.0, neginf=0.0)
        return dict(
            in_weights=_get(self.in_weights),
            hid_weights=_get(self.hid_weights),
            input_activity=_get(self.input_activity),
            hidden_activity=_get(self.hidden_activity),
        )


def _report_nonfinite(bank, name):
    """One-time NaN/Inf scan over a fingerprint bank so a corrupted upstream
    STDP run is visible in the logs rather than only showing up as `loss=nan`
    several minutes into training."""
    parts = dict(in_weights=bank.in_weights, hid_weights=bank.hid_weights,
                 input_activity=bank.input_activity, hidden_activity=bank.hidden_activity)
    any_bad = False
    for key, arr in parts.items():
        a = np.asarray(arr)
        n_nan, n_inf = int(np.isnan(a).sum()), int(np.isinf(a).sum())
        if n_nan or n_inf:
            any_bad = True
            print(f"[fp-check][{name}] {key}: {n_nan} NaN, {n_inf} Inf "
                  f"out of {a.size} values")
    print(f"[fp-check][{name}] {'found non-finite values above — ' if any_bad else 'clean — '}"
          f"batches replace any NaN/Inf with 0.0")


train_fp_prefix, train_fp_meta = build_fingerprint_cache(FINGERPRINT_TRAIN_GLOB, "train")
test_fp_prefix,  test_fp_meta  = build_fingerprint_cache(FINGERPRINT_TEST_GLOB,  "test")
train_fp = FingerprintBank(train_fp_prefix, train_fp_meta)
test_fp  = FingerprintBank(test_fp_prefix,  test_fp_meta)
print(f"[fp] train {train_fp.n} fingerprints | test {test_fp.n} fingerprints")
_report_nonfinite(train_fp, "train")
_report_nonfinite(test_fp, "test")

In [ ]:
# ═════════════════════════════════════════════════════════════════════════════
#  ECAPA-TDNN — ported verbatim from https://github.com/TaoRuijie/ECAPA-TDNN
#  (itself derived from clovaai/voxceleb_trainer, lawlict/ECAPA-TDNN, speechbrain)
#  Deviations: fbank front-end AND attentive-stats pooling forced to fp32 (both
#  overflow/underflow under AMP); no SpecAugment (no augmentation at all — see
#  CONFIG cell); optional FiLM conditioning on the post-MFA feature map from an
#  SNN fingerprint.
#  Defined here (before the decode cell) so the sanity-check cell right after
#  this one can smoke-test the model with dummy audio + real fingerprints
#  before paying for the full audio decode.
# ═════════════════════════════════════════════════════════════════════════════
import math
import torch
import torchaudio
import torch.nn as nn
import torch.nn.functional as F

class SEModule(nn.Module):
    def __init__(self, channels, bottleneck=128):
        super().__init__()
        self.se = nn.Sequential(
            nn.AdaptiveAvgPool1d(1),
            nn.Conv1d(channels, bottleneck, kernel_size=1, padding=0),
            nn.ReLU(),
            nn.Conv1d(bottleneck, channels, kernel_size=1, padding=0),
            nn.Sigmoid())

    def forward(self, x):
        return x * self.se(x)

class Bottle2neck(nn.Module):
    def __init__(self, inplanes, planes, kernel_size=None, dilation=None, scale=8):
        super().__init__()
        width = int(math.floor(planes / scale))
        self.conv1 = nn.Conv1d(inplanes, width * scale, kernel_size=1)
        self.bn1   = nn.BatchNorm1d(width * scale)
        self.nums  = scale - 1
        num_pad = math.floor(kernel_size / 2) * dilation
        self.convs = nn.ModuleList([
            nn.Conv1d(width, width, kernel_size=kernel_size, dilation=dilation,
                      padding=num_pad) for _ in range(self.nums)])
        self.bns   = nn.ModuleList([nn.BatchNorm1d(width) for _ in range(self.nums)])
        self.conv3 = nn.Conv1d(width * scale, planes, kernel_size=1)
        self.bn3   = nn.BatchNorm1d(planes)
        self.relu  = nn.ReLU()
        self.width = width
        self.se    = SEModule(planes)

    def forward(self, x):
        residual = x
        out = self.bn1(self.relu(self.conv1(x)))
        spx = torch.split(out, self.width, 1)
        for i in range(self.nums):
            sp = spx[i] if i == 0 else sp + spx[i]
            sp = self.bns[i](self.relu(self.convs[i](sp)))
            out = sp if i == 0 else torch.cat((out, sp), 1)
        out = torch.cat((out, spx[self.nums]), 1)
        out = self.bn3(self.relu(self.conv3(out)))
        return self.se(out) + residual

class PreEmphasis(nn.Module):
    def __init__(self, coef=0.97):
        super().__init__()
        self.coef = coef
        self.register_buffer("flipped_filter",
                             torch.FloatTensor([-coef, 1.]).unsqueeze(0).unsqueeze(0))

    def forward(self, x):
        x = F.pad(x.unsqueeze(1), (1, 0), "reflect")
        return F.conv1d(x, self.flipped_filter).squeeze(1)


class FingerprintFiLM(nn.Module):
    """STDP fingerprint -> per-channel (gamma, beta) for the post-MFA feature map.

    Each fingerprint component gets its own small encoder branch (LayerNorm ->
    Linear -> ReLU). A component whose FINGERPRINT_PARTS flag is off contributes a
    constant near-zero vector (`eps`, not an exact zero) instead of running its
    encoder — the shared trunk's input size and total parameter count are
    identical in every ablation config, so an EER difference is attributable only
    to the missing signal, never to a smaller model.

    gamma/beta heads are zero-initialised, so FiLM starts as an exact identity
    transform (gamma=1, beta=0). gamma/beta are additionally bounded via tanh
    (gamma in [1-gamma_range, 1+gamma_range], beta in [-beta_scale, beta_scale])
    so the modulation cannot grow without limit — tanh(0)=0 at init, so the
    identity start is exact regardless of the bound. Note this bound alone does
    NOT make the model AMP-safe: the fp16 overflow that produced NaN lived in the
    attentive-stats pooling (x**2), which ECAPA_TDNN.forward now runs in fp32.
    """
    _DIMS = dict(in_weights=2 * 23 * 128, hid_weights=2 * 23 * 128,
                 input_activity=128, hidden_activity=128)

    def __init__(self, film_channels, branch_dim, hidden, parts, eps, gamma_range, beta_scale):
        super().__init__()
        self.parts = dict(parts)
        self.eps = eps
        self.branch_dim = branch_dim
        self.gamma_range = gamma_range
        self.beta_scale = beta_scale
        self.norms    = nn.ModuleDict({k: nn.LayerNorm(d) for k, d in self._DIMS.items()})
        self.encoders = nn.ModuleDict({k: nn.Linear(d, branch_dim) for k, d in self._DIMS.items()})
        self.trunk    = nn.Sequential(nn.Linear(branch_dim * len(self._DIMS), hidden), nn.ReLU())
        self.to_gamma = nn.Linear(hidden, film_channels)
        self.to_beta  = nn.Linear(hidden, film_channels)
        nn.init.zeros_(self.to_gamma.weight); nn.init.zeros_(self.to_gamma.bias)
        nn.init.zeros_(self.to_beta.weight);  nn.init.zeros_(self.to_beta.bias)

    def forward(self, fp):
        branches = []
        for key in self._DIMS:
            x = fp[key]
            if self.parts.get(key, True):
                b = F.relu(self.encoders[key](self.norms[key](x.flatten(1))))
            else:
                b = torch.full((x.shape[0], self.branch_dim), self.eps,
                               device=x.device, dtype=x.dtype)
            branches.append(b)
        h = self.trunk(torch.cat(branches, dim=1))
        gamma = 1.0 + self.gamma_range * torch.tanh(self.to_gamma(h))
        beta  = self.beta_scale * torch.tanh(self.to_beta(h))
        return gamma, beta


class ECAPA_TDNN(nn.Module):
    def __init__(self, C, use_film=False, film_parts=None):
        super().__init__()
        self.torchfbank = nn.Sequential(
            PreEmphasis(),
            torchaudio.transforms.MelSpectrogram(
                sample_rate=16000, n_fft=512, win_length=400, hop_length=160,
                f_min=20, f_max=7600, window_fn=torch.hamming_window, n_mels=80))

        self.conv1  = nn.Conv1d(80, C, kernel_size=5, stride=1, padding=2)
        self.relu   = nn.ReLU()
        self.bn1    = nn.BatchNorm1d(C)
        self.layer1 = Bottle2neck(C, C, kernel_size=3, dilation=2, scale=8)
        self.layer2 = Bottle2neck(C, C, kernel_size=3, dilation=3, scale=8)
        self.layer3 = Bottle2neck(C, C, kernel_size=3, dilation=4, scale=8)
        self.layer4 = nn.Conv1d(3 * C, 1536, kernel_size=1)
        self.attention = nn.Sequential(
            nn.Conv1d(4608, 256, kernel_size=1), nn.ReLU(), nn.BatchNorm1d(256),
            nn.Tanh(), nn.Conv1d(256, 1536, kernel_size=1), nn.Softmax(dim=2))
        self.bn5 = nn.BatchNorm1d(3072)
        self.fc6 = nn.Linear(3072, 192)
        self.bn6 = nn.BatchNorm1d(192)

        self.use_film = use_film
        self.film = FingerprintFiLM(film_channels=1536, branch_dim=FILM_BRANCH_DIM,
                                    hidden=FILM_HIDDEN, parts=film_parts or {},
                                    eps=FILM_DISABLED_EPS, gamma_range=FILM_GAMMA_RANGE,
                                    beta_scale=FILM_BETA_SCALE) if use_film else None

    def forward(self, x, fp=None):
        # fbank in fp32 regardless of the surrounding autocast: log() of a fp16
        # mel spectrogram underflows for quiet frames.
        with torch.no_grad(), torch.autocast("cuda", enabled=False):
            x = self.torchfbank(x.float()) + 1e-6
            x = x.log()
            x = x - torch.mean(x, dim=-1, keepdim=True)

        x = self.bn1(self.relu(self.conv1(x)))
        x1 = self.layer1(x)
        x2 = self.layer2(x + x1)
        x3 = self.layer3(x + x1 + x2)
        x  = self.relu(self.layer4(torch.cat((x1, x2, x3), dim=1)))

        if self.film is not None:
            assert fp is not None, "fp is required when use_film=True"
            gamma, beta = self.film(fp)
            x = gamma.unsqueeze(-1) * x + beta.unsqueeze(-1)

        # Attentive statistics pooling in fp32. The post-MFA map is unnormalised
        # (there is no BatchNorm after layer4), so |x| in the tens-to-hundreds is
        # normal — and both x**2 here and var(x) below overflow fp16 once
        # |x| > 256, making `inf - inf` -> NaN. FiLM reaches that threshold sooner
        # because gamma scales x by up to 1+FILM_GAMMA_RANGE, which is why the
        # NaN showed up only with USE_FILM=True. Once it happens GradScaler skips
        # every step, so the weights can never shrink back and training deadlocks
        # at loss=nan permanently — hence fp32 here rather than a looser bound.
        with torch.autocast("cuda", enabled=False):
            x = x.float()
            t = x.size()[-1]
            global_x = torch.cat((
                x,
                torch.mean(x, dim=2, keepdim=True).repeat(1, 1, t),
                torch.sqrt(torch.var(x, dim=2, keepdim=True).clamp(min=1e-4)).repeat(1, 1, t)),
                dim=1)
            w  = self.attention(global_x)
            mu = torch.sum(x * w, dim=2)
            sg = torch.sqrt((torch.sum((x ** 2) * w, dim=2) - mu ** 2).clamp(min=1e-4))
            return self.bn6(self.fc6(self.bn5(torch.cat((mu, sg), 1))))

_m = ECAPA_TDNN(C, use_film=USE_FILM, film_parts=FINGERPRINT_PARTS)
_n_film = sum(p.numel() for p in (_m.film.parameters() if _m.film is not None else []))
print(f"ECAPA-TDNN C={C}: "
      f"{sum(p.numel() for p in _m.parameters() if p.requires_grad)/1e6:.2f}M params "
      f"({_n_film/1e3:.0f}K in FiLM)")
del _m

In [ ]:
# ── AAMSoftmax (same formulation as the reference loss.py and ann_backend.ipynb) ──
class AAMSoftmax(nn.Module):
    def __init__(self, embed_dim, num_classes, m=AAM_M, s=AAM_S):
        super().__init__()
        self.s = s
        self.W = nn.Parameter(torch.empty(num_classes, embed_dim))
        nn.init.xavier_normal_(self.W)
        self.m = m
        self.cos_m, self.sin_m = math.cos(m), math.sin(m)
        self.th = math.cos(math.pi - m)
        self.mm = math.sin(math.pi - m) * m

    def forward(self, emb, labels):
        cos = F.linear(F.normalize(emb), F.normalize(self.W)).clamp(-1 + 1e-7, 1 - 1e-7)
        sin = torch.sqrt((1 - cos ** 2).clamp_min(1e-9))
        phi = cos * self.cos_m - sin * self.sin_m
        phi = torch.where(cos > self.th, phi, cos - self.mm)
        one_hot = torch.zeros_like(cos).scatter_(1, labels.view(-1, 1), 1.0)
        logits = (one_hot * phi + (1 - one_hot) * cos) * self.s
        return F.cross_entropy(logits, labels)

In [ ]:
# ═════════════════════════════════════════════════════════════════════════════
#  SANITY CHECK — dummy forward/backward pass BEFORE paying for the ~2hr decode
# ═════════════════════════════════════════════════════════════════════════════
# Catches wiring bugs (shape mismatches, NaN/Inf fingerprints breaking FiLM's
# zero-init identity trick, AAMSoftmax misconfiguration, ...) in seconds using
# dummy random audio (real audio isn't decoded yet) + a few real fingerprint
# rows (train_fp is already loaded, cheap). This is exactly the check that
# would have caught the FiLM NaN bug before the full decode+epoch instead of
# after it.
def sanity_check():
    net = ECAPA_TDNN(C, use_film=USE_FILM, film_parts=FINGERPRINT_PARTS).to(device)
    n = 4
    n_classes = max(2, len(set(train_fp.person_ids[:n].tolist())) + 1)
    aam = AAMSoftmax(EMBED_DIM, n_classes).to(device)

    x = torch.randn(n, CROP_SAMPLES, device=device)              # dummy audio
    fp = train_fp.batch(np.arange(n), device) if USE_FILM else None  # real fingerprints
    y = torch.randint(0, n_classes, (n,), device=device)

    net.train()
    emb = net(x, fp=fp)
    loss = aam(emb.float(), y)
    loss.backward()

    assert torch.isfinite(emb).all(), "sanity check FAILED: embeddings contain NaN/Inf"
    assert torch.isfinite(loss), f"sanity check FAILED: loss is {loss.item()}"
    bad_grads = [name for name, p in list(net.named_parameters()) + list(aam.named_parameters())
                 if p.grad is not None and not torch.isfinite(p.grad).all()]
    assert not bad_grads, f"sanity check FAILED: non-finite gradients in {bad_grads[:5]}"

    print(f"[sanity] forward+backward OK — loss={loss.item():.4f}, "
          f"embeddings and all gradients finite. Safe to proceed to decode.")
    del net, aam

sanity_check()

In [ ]:
# ── Resolve each fingerprint's audio, then decode once -> ragged int16 memmap ──
# Fingerprints (not an independent corpus walk) drive enumeration: every clip
# trained/evaluated on is one prepare_fingerprints.ipynb already fingerprinted,
# located here purely by (person_id, session_id, file_name).
from concurrent.futures import ThreadPoolExecutor
import shutil, subprocess
import soundfile as sf

def _build_person_index(glob_pattern, dev_nn_style):
    """{person_id -> base_dir} where base_dir/{person_id}/{session_id}/{file}
    exists. dev_nn_style=True adds VoxCeleb1's extra dev_NN level (root/dev_NN/
    person/session/wav) — cheap directory listings only, no recursive file walk."""
    index = {}
    for root in sorted(glob.glob(glob_pattern)):
        bases = sorted(glob.glob(os.path.join(root, "dev_*"))) if dev_nn_style else [root]
        for base in bases:
            try:
                persons = os.listdir(base)
            except OSError:
                continue
            for p in persons:
                index.setdefault(p, base)
    return index


def resolve_entries(fp_bank, glob_pattern, name, dev_nn_style):
    """Build each fingerprint's expected wav path from the cheap person_index —
    no per-row os.path.exists() (172k+ individual stat calls against a Kaggle
    dataset mount cost ~17min doing that). A genuinely missing/corrupt file is
    still caught, just later: build_cache's decode step already treats a failed
    ffmpeg/soundfile read as `None` and counts it in n_fail."""
    person_index = _build_person_index(glob_pattern, dev_nn_style)
    entries, missing = [], 0
    for i in range(fp_bank.n):
        pid, sid, fname = fp_bank.person_ids[i], fp_bank.session_ids[i], fp_bank.file_names[i]
        base = person_index.get(pid)
        if base is None:
            missing += 1
            continue
        path = os.path.join(base, pid, sid, fname)
        entries.append(dict(person_id=pid, record_id=sid, wav_path=path, fp_row=i))
    print(f"[{name}] resolved {len(entries)}/{fp_bank.n} fingerprints to audio "
          f"({missing} with no known speaker directory; existence of the exact "
          f"file is verified during decode)")
    assert entries, f"[{name}] no fingerprinted clip's audio could be found"
    return entries

train_entries = resolve_entries(train_fp, TRAIN_GLOB, "train/vox2", dev_nn_style=False)
test_entries  = resolve_entries(test_fp,  TEST_GLOB,  "eval/vox1",  dev_nn_style=True)

_dev_spk  = {e["person_id"] for e in train_entries}
_test_spk = {e["person_id"] for e in test_entries}
_overlap  = _dev_spk & _test_spk
assert not _overlap, f"train/eval speaker overlap: {sorted(_overlap)[:10]}"
print(f"[disjoint] train={len(_dev_spk)}  eval={len(_test_spk)}  overlap=0")


def _decode_ffmpeg(path, sr, max_sec=None):
    """`-t max_sec` stops ffmpeg decoding once it has enough output instead of
    decoding the whole file and truncating in Python afterward — since every
    clip here is capped to CLIP_SEC anyway, this is the difference between
    decoding ~2s and decoding the full (often much longer) source file."""
    cmd = ["ffmpeg", "-nostdin", "-loglevel", "error", "-i", path]
    if max_sec is not None:
        cmd += ["-t", f"{max_sec:.3f}"]
    cmd += ["-ac", "1", "-ar", str(sr), "-f", "f32le", "-acodec", "pcm_f32le", "-"]
    proc = subprocess.run(cmd, capture_output=True)
    if proc.returncode != 0:
        raise RuntimeError(proc.stderr.decode("utf-8", "ignore")[:200])
    return np.frombuffer(proc.stdout, dtype="<f4").astype(np.float32)

_FFMPEG_EXTS = (".m4a", ".aac", ".mp4", ".m4b")

def load_audio(path, sr=SR, max_sec=None):
    """Mono float32 at `sr`. Both VoxCeleb1 and VoxCeleb2 are m4a here, so ffmpeg
    is the normal path and is dispatched on extension — a try/except around
    soundfile would raise once per file across the whole corpus."""
    if path.lower().endswith(_FFMPEG_EXTS):
        return _decode_ffmpeg(path, sr, max_sec)
    try:
        y, sr_native = sf.read(path, dtype="float32", always_2d=False)
    except Exception:
        return _decode_ffmpeg(path, sr, max_sec)
    if y.ndim > 1:
        y = y.mean(axis=1)
    if sr_native != sr:
        import librosa
        y = librosa.resample(y, orig_sr=sr_native, target_sr=sr)
    return np.ascontiguousarray(y, dtype=np.float32)

def _decode_one(args):
    path, cap = args
    try:
        y = load_audio(path, SR, max_sec=cap / SR)
    except Exception:
        return None
    if y.size == 0:
        return None
    if y.size > cap:                 # ffmpeg's -t is a soft bound; trim any excess
        y = y[:cap]
    return np.clip(y * 32768.0, -32768, 32767).astype(np.int16)

def build_cache(entries, name, cap_sec, workers=None):
    """Decode `entries` (each carrying fp_row) into {CACHE_DIR}/{name}.bin +
    {name}_index.npz, capped to CLIP_SEC — only the first 2s of every clip is
    ever decoded or stored. Idempotent."""
    cap = int(cap_sec * SR)
    bin_path = os.path.join(CACHE_DIR, f"{name}.bin")
    idx_path = os.path.join(CACHE_DIR, f"{name}_index.npz")

    if os.path.exists(bin_path) and os.path.exists(idx_path):
        z = np.load(idx_path, allow_pickle=True)
        if int(z["n_entries"]) == len(entries) and int(z["cap"]) == cap \
           and os.path.getsize(bin_path) == int(z["nbytes"]):
            print(f"[{name}] cache hit: {int(z['n_kept'])} clips, "
                  f"{int(z['nbytes'])/1e9:.2f} GB — skipping decode")
            return idx_path
        print(f"[{name}] cache stale (entries/cap/size mismatch) — rebuilding")

    est  = len(entries) * cap * 2 * 0.92
    free = shutil.disk_usage(CACHE_DIR).free
    print(f"[{name}] ~{est/1e9:.1f} GB estimated, {free/1e9:.1f} GB free on {CACHE_DIR}")
    if est > free:
        raise RuntimeError(
            f"[{name}] not enough scratch space: need ~{est/1e9:.1f} GB, have "
            f"{free/1e9:.1f} GB. Point CACHE_DIR at a bigger volume.")

    workers = workers or (os.cpu_count() or 2) * 3
    paths = [e["wav_path"] for e in entries]
    offsets = np.zeros(len(paths), dtype=np.int64)
    lengths = np.zeros(len(paths), dtype=np.int64)

    t0, pos, n_fail, i = time.time(), 0, 0, 0
    CHUNK = 512
    with open(bin_path, "wb", buffering=1 << 22) as f, \
         ThreadPoolExecutor(max_workers=workers) as pool:
        for s in range(0, len(paths), CHUNK):
            batch = paths[s:s + CHUNK]
            for arr in pool.map(_decode_one, [(p, cap) for p in batch]):
                if arr is None:
                    n_fail += 1
                    offsets[i], lengths[i] = pos, 0
                else:
                    f.write(arr.tobytes())
                    offsets[i], lengths[i] = pos, arr.size
                    pos += arr.size
                i += 1
            if s % (CHUNK * 20) == 0 and s:
                el = time.time() - t0
                print(f"  [{name}] {i}/{len(paths)}  {pos*2/1e9:.2f} GB  "
                      f"{el:.0f}s  eta {el/i*(len(paths)-i):.0f}s", flush=True)

    keep = np.nonzero(lengths)[0]
    np.savez(idx_path,
             offsets=offsets[keep], lengths=lengths[keep],
             person_ids=np.array([entries[i]["person_id"] for i in keep]),
             record_ids=np.array([entries[i]["record_id"] for i in keep]),
             fp_row=np.array([entries[i]["fp_row"] for i in keep], dtype=np.int64),
             n_entries=len(entries), n_kept=len(keep), cap=cap, nbytes=pos * 2)
    print(f"[{name}] decoded {len(keep)}/{len(paths)} clips ({n_fail} failed) "
          f"-> {pos*2/1e9:.2f} GB in {time.time()-t0:.0f}s")
    return idx_path

train_idx = build_cache(train_entries, "train", TRAIN_CAP_SEC)
test_idx  = build_cache(test_entries,  "test",  TEST_CAP_SEC)

In [ ]:
# ── Waveform bank ─────────────────────────────────────────────────────────────
# No DataLoader: reads are memmap slices out of the page cache, so a plain bank
# with .batch(idx) is both faster and free of the worker-caching leak that bit
# the fingerprint backend. No augmentation (see CONFIG cell) — every clip is the
# same clean, continuous CLIP_SEC window every epoch, matching what the FiLM
# fingerprint was computed from.

def encode_labels(pid):
    classes = sorted(set(pid.tolist()))
    c2i = {c: i for i, c in enumerate(classes)}
    return np.fromiter((c2i[p] for p in pid), dtype=np.int64, count=len(pid)), len(classes)

class WaveBank:
    """Ragged int16 memmap -> float32 crops, plus each clip's fingerprint row."""
    def __init__(self, bin_path, idx_path):
        z = np.load(idx_path, allow_pickle=True)
        self.offsets = z["offsets"]; self.lengths = z["lengths"]
        self.person_ids = z["person_ids"].astype(str)
        self.record_ids = z["record_ids"].astype(str)
        self.fp_row = z["fp_row"]
        self.wav = np.memmap(bin_path, dtype=np.int16, mode="r")
        self.n = len(self.offsets)

    def raw(self, i, cap=None):
        o, L = int(self.offsets[i]), int(self.lengths[i])
        if cap is not None:
            L = min(L, cap)
        return np.asarray(self.wav[o:o + L], dtype=np.float32) / 32768.0

    def crop(self, i, n):
        """Random n-sample crop; wrap-pad clips shorter than n (voxceleb_trainer).
        In practice a no-op here since the stored length already equals n (both
        capped to CLIP_SEC) — kept for robustness against any rounding shortfall."""
        o, L = int(self.offsets[i]), int(self.lengths[i])
        if L < n:
            y = np.asarray(self.wav[o:o + L], dtype=np.float32) / 32768.0
            return np.pad(y, (0, n - L), mode="wrap")
        s = random.randint(0, L - n)
        return np.asarray(self.wav[o + s:o + s + n], dtype=np.float32) / 32768.0

    def batch(self, idx, n=CROP_SAMPLES):
        out = np.empty((len(idx), n), dtype=np.float32)
        for k, i in enumerate(idx):
            out[k] = self.crop(int(i), n)
        return torch.from_numpy(out)

train_bank = WaveBank(os.path.join(CACHE_DIR, "train.bin"), train_idx)
test_bank  = WaveBank(os.path.join(CACHE_DIR, "test.bin"),  test_idx)

y_train, NUM_CLASSES = encode_labels(train_bank.person_ids)
y_test,  _           = encode_labels(test_bank.person_ids)
print(f"[bank] train {train_bank.n} clips / {NUM_CLASSES} speakers | "
      f"eval {test_bank.n} clips / {len(set(y_test.tolist()))} speakers")

In [ ]:
# ═════════════════════════════════════════════════════════════════════════════
#  EVAL — same protocol as ann_backend.ipynb (session-free all-pairs retrieval)
#  Two changes for the large gallery: vectorized AP (identical math, no per-row
#  GPU sync) and length-bucketed embedding extraction.
# ═════════════════════════════════════════════════════════════════════════════
@torch.no_grad()
def extract_embeddings(net, bank, fp_bank, cap_sec=TEST_CAP_SEC):
    """Embeddings from up to the first cap_sec of each clip (here: CLIP_SEC for
    every clip, so effectively the whole stored clip). Length-sorted buckets
    truncated to the batch minimum: no padding, so ASP attention statistics stay
    exact. fp_bank supplies FiLM's fingerprint per clip via bank.fp_row; pass
    fp_bank=None to run without FiLM regardless of USE_FILM."""
    net.eval()
    cap = int(cap_sec * SR)
    lens = np.minimum(bank.lengths, cap)
    order = np.argsort(lens)                      # ascending, so trim within a batch is tiny
    out = torch.empty(bank.n, EMBED_DIM, dtype=torch.float32)
    for s in range(0, bank.n, EVAL_BATCH):
        rows = order[s:s + EVAL_BATCH]
        n = int(lens[rows].min())
        n = max(n, CROP_SAMPLES)                  # guard: never shorter than one crop
        batch = np.stack([bank.crop(int(i), n) if lens[i] < n else bank.raw(int(i), cap)[:n]
                          for i in rows])
        x = torch.from_numpy(batch).to(device)
        fp = fp_bank.batch(bank.fp_row[rows], device) if fp_bank is not None else None
        with torch.autocast("cuda", dtype=torch.float16):
            e = net(x, fp=fp)
        out[torch.from_numpy(rows)] = F.normalize(e.float(), dim=1).cpu()
    return out

def _eer_from_hist(gen_hist, imp_hist):
    g  = gen_hist / max(gen_hist.sum(), 1)
    im = imp_hist / max(imp_hist.sum(), 1)
    frr = np.cumsum(g)               # genuine in bins <= thr -> rejected
    far = 1.0 - np.cumsum(im)        # impostor in bins  > thr -> accepted
    k = int(np.argmin(np.abs(frr - far)))
    return float((frr[k] + far[k]) / 2)

@torch.no_grad()
def retrieval_metrics(embs, labels, record_ids, session_free=True, block=512, n_bins=20000):
    N = embs.shape[0]
    E   = embs.to(device)
    lab = torch.as_tensor(labels, device=device)
    rec = torch.as_tensor(np.unique(record_ids, return_inverse=True)[1], device=device)
    gen_hist = np.zeros(n_bins); imp_hist = np.zeros(n_bins)
    r1 = r5 = 0; ap_sum = 0.0; valid = 0
    ranks = torch.arange(1, N + 1, device=device, dtype=torch.float32).view(1, -1)

    for s in range(0, N, block):
        e    = E[s:s + block]
        sims = e @ E.t()
        b    = sims.shape[0]
        rows = torch.arange(s, s + b, device=device)
        if session_free:
            excl = rec[rows][:, None] == rec[None, :]     # same recording (incl. self)
        else:
            excl = rows[:, None] == torch.arange(N, device=device)[None, :]
        sims = sims.masked_fill(excl, -2.0)
        same = (lab[rows][:, None] == lab[None, :]) & (~excl)

        order = torch.argsort(sims, dim=1, descending=True)
        same_sorted = torch.gather(same, 1, order)
        r1 += same_sorted[:, 0].sum().item()
        r5 += same_sorted[:, :5].any(dim=1).sum().item()

        # Vectorized AP: mean over hits of precision@rank. Identical to the
        # per-row loop in ann_backend.ipynb, without the per-row sync.
        rel  = same.sum(dim=1)
        hits = torch.cumsum(same_sorted.float(), dim=1)
        prec = hits / ranks
        ap   = (prec * same_sorted.float()).sum(dim=1) / rel.clamp(min=1)
        ok   = rel > 0
        ap_sum += float(ap[ok].sum()); valid += int(ok.sum())

        gen_hist += torch.histc(sims[same],              bins=n_bins, min=-1, max=1).cpu().numpy()
        imp_hist += torch.histc(sims[(~same) & (~excl)], bins=n_bins, min=-1, max=1).cpu().numpy()

    return dict(rank1=r1 / max(valid, 1), rank5=r5 / max(valid, 1),
                mAP=ap_sum / max(valid, 1), eer=_eer_from_hist(gen_hist, imp_hist))

def evaluate(net, bank, labels, fp_bank):
    embs = extract_embeddings(net, bank, fp_bank)
    return retrieval_metrics(embs, labels, bank.record_ids, session_free=True)

In [ ]:
# ═════════════════════════════════════════════════════════════════════════════
#  TRAIN
# ═════════════════════════════════════════════════════════════════════════════
def rss_gb():
    with open("/proc/self/statm") as f:
        return int(f.read().split()[1]) * os.sysconf("SC_PAGE_SIZE") / 1e9

def banner():
    print("===== RUN CONFIG =====")
    print(f"  ECAPA-TDNN C={C}  embed={EMBED_DIM}  batch={BATCH_SIZE}  epochs={EPOCHS}")
    print(f"  Adam lr={LR} wd={WEIGHT_DECAY} decay={LR_DECAY}/epoch | AAM m={AAM_M} s={AAM_S}")
    print(f"  clip={CLIP_SEC}s (train+eval)  augmentation=none")
    print(f"  USE_FILM={USE_FILM}  parts={FINGERPRINT_PARTS if USE_FILM else '-'}")
    print(f"  train {train_bank.n} clips / {NUM_CLASSES} spk | eval {test_bank.n} clips")
    print("======================")

def train():
    net = ECAPA_TDNN(C, use_film=USE_FILM, film_parts=FINGERPRINT_PARTS).to(device)
    aam = AAMSoftmax(EMBED_DIM, NUM_CLASSES).to(device)
    opt = torch.optim.Adam(list(net.parameters()) + list(aam.parameters()),
                           lr=LR, weight_decay=WEIGHT_DECAY)
    sched  = torch.optim.lr_scheduler.ExponentialLR(opt, gamma=LR_DECAY)
    scaler = torch.amp.GradScaler("cuda")

    history, best_eer, start_epoch = [], float("inf"), 0
    banner()

    if RESUME and os.path.exists(LAST_CKPT):
        ck = torch.load(LAST_CKPT, map_location=device)
        if ck.get("tag") == TAG:
            net.load_state_dict(ck["model"]); aam.load_state_dict(ck["aam"])
            opt.load_state_dict(ck["opt"]);   sched.load_state_dict(ck["sched"])
            scaler.load_state_dict(ck["scaler"])
            history, best_eer, start_epoch = ck["history"], ck["best_eer"], ck["epoch"] + 1
            print(f"[resume] epoch {start_epoch}  best_eer={best_eer:.4f}")
        else:
            print(f"[resume] tag mismatch ({ck.get('tag')} != {TAG}); fresh start")

    y_t = torch.from_numpy(y_train)
    steps = train_bank.n // BATCH_SIZE
    rng = np.random.default_rng(SEED + start_epoch)

    for epoch in range(start_epoch, EPOCHS):
        net.train()
        perm = rng.permutation(train_bank.n)
        t0, running, nb, n_bad = time.time(), 0.0, 0, 0

        for si in range(steps):
            idx = perm[si * BATCH_SIZE:(si + 1) * BATCH_SIZE]
            x = train_bank.batch(idx).to(device, non_blocking=True)
            y = y_t[idx].to(device, non_blocking=True)
            fp = train_fp.batch(train_bank.fp_row[idx], device) if USE_FILM else None

            opt.zero_grad(set_to_none=True)
            with torch.autocast("cuda", dtype=torch.float16):
                emb = net(x, fp=fp)
            loss = aam(emb.float(), y)                      # AAM in fp32

            # A non-finite loss would poison the running average for the whole
            # epoch (and GradScaler would skip the step anyway), so drop the
            # batch and count it instead — n_bad in the epoch line shows whether
            # this is a rare transient or the permanent NaN deadlock.
            if not torch.isfinite(loss):
                n_bad += 1
                if n_bad == 1:
                    print(f"  [e{epoch:02d} {si+1}/{steps}] WARNING non-finite loss "
                          f"— batch skipped", flush=True)
                continue

            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(
                list(net.parameters()) + list(aam.parameters()), GRAD_CLIP)
            scaler.step(opt); scaler.update()
            running += loss.item(); nb += 1

            if (si + 1) % 200 == 0:
                print(f"  [e{epoch:02d} {si+1}/{steps}] loss={running/max(nb,1):.4f} "
                      f"({time.time()-t0:.0f}s)", flush=True)
        sched.step()

        rec = dict(epoch=epoch, loss=running / max(nb, 1), n_bad=n_bad,
                   lr=opt.param_groups[0]["lr"], rss=rss_gb())
        bad_str = f" bad={n_bad}/{steps}" if n_bad else ""
        improved = False
        if (epoch + 1) % EVAL_EVERY == 0 or epoch == EPOCHS - 1:
            m = evaluate(net, test_bank, y_test, test_fp if USE_FILM else None)
            rec["metrics"] = m
            improved = m["eer"] < best_eer
            if improved:
                best_eer = m["eer"]
                torch.save(dict(tag=TAG, epoch=epoch, metrics=m,
                                model=net.state_dict(), aam=aam.state_dict()), BEST_CKPT)
            print(f"[e{epoch:02d}] loss={rec['loss']:.4f} lr={rec['lr']:.2e}{bad_str} | "
                  f"eer={m['eer']:.4f} r1={m['rank1']:.3f} r5={m['rank5']:.3f} "
                  f"mAP={m['mAP']:.3f} | RSS={rec['rss']:.1f}GB "
                  f"{'*best' if improved else ''} ({time.time()-t0:.0f}s)")
        else:
            print(f"[e{epoch:02d}] loss={rec['loss']:.4f} lr={rec['lr']:.2e}{bad_str} | "
                  f"RSS={rec['rss']:.1f}GB ({time.time()-t0:.0f}s)")

        history.append(rec)
        torch.save(dict(tag=TAG, epoch=epoch, best_eer=best_eer, history=history,
                        model=net.state_dict(), aam=aam.state_dict(),
                        opt=opt.state_dict(), sched=sched.state_dict(),
                        scaler=scaler.state_dict()), LAST_CKPT)

    print(f"[done] best EER={best_eer:.4f}  ->  {BEST_CKPT}")
    return history

In [ ]:
# ===================== RUN =====================
history = train()

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  TEMPORARY PROBE — does the SNN fingerprint carry speaker identity AT ALL?
#  Self-contained. Delete this cell once the question is answered; nothing else
#  in the notebook reads anything it defines (every name here is PROBE_*/_p*).
#
#  Runs THIS notebook's own eval protocol (session-free all-pairs retrieval,
#  EER / Rank-1 / Rank-5 / mAP) directly on the raw fingerprints with **L1
#  distance** — no ECAPA, no training, no audio. If the fingerprints alone are at
#  chance for speaker retrieval, the ~1% FiLM delta is not identity information
#  and no number of seeds will change that. Also breaks the answer down per
#  component, which is the 4-run ablation for free.
#
#  PREREQUISITES: only the CONFIG cell and the fingerprint-cache cell (they define
#  `test_fp` / `train_fp` / `device`). The audio-decode, WaveBank and train cells
#  are NOT needed — skip them and jump straight here.
#
#  Reference points printed below: EER 0.5 = chance; the `random` control row is
#  an empirical floor (Gaussian noise features, same protocol, same N).
# ══════════════════════════════════════════════════════════════════════════════
import numpy as np, torch, time

PROBE_BANK      = test_fp    # vox1 eval fingerprints (speakers unseen by any training)
PROBE_MAX_CLIPS = 5000       # random subsample; None = all 22496 (cost is O(N^2*D),
                             #  and the N x N score matrix must fit in GPU memory:
                             #  5000 -> 100 MB, 22496 -> 2.0 GB)
PROBE_SEED      = 0
PROBE_ROW_BLOCK = 128

# Each component on its own, then all four concatenated. "l1-normalised" divides
# every vector by its own L1 mass first: without it, raw L1 distance is dominated
# by how much total weight/activity a sample has (silent-cell zeroing makes that
# vary a lot between clips), which would measure loudness, not identity.
PROBE_CONFIGS = [
    ("in_weights",      False), ("in_weights",      True),
    ("hid_weights",     False), ("hid_weights",     True),
    ("input_activity",  False), ("input_activity",  True),
    ("hidden_activity", False), ("hidden_activity", True),
    ("all4",            True),
    ("random",          False),   # control: Gaussian noise -> expect EER ~ 0.5
]

# ── Sample selection + protocol masks ─────────────────────────────────────────
_prng = np.random.default_rng(PROBE_SEED)
_n_all = PROBE_BANK.n
if PROBE_MAX_CLIPS is None or PROBE_MAX_CLIPS >= _n_all:
    _rows = np.arange(_n_all)
else:
    _rows = np.sort(_prng.choice(_n_all, PROBE_MAX_CLIPS, replace=False))
_N = len(_rows)

_pid = PROBE_BANK.person_ids[_rows]
_sid = PROBE_BANK.session_ids[_rows]
_lab = torch.as_tensor(np.unique(_pid, return_inverse=True)[1], device=device)
_rid = torch.as_tensor(
    np.unique(np.char.add(np.char.add(_pid, "/"), _sid), return_inverse=True)[1],
    device=device)

# Same exclusion rule as retrieval_metrics(session_free=True): drop same-recording
# pairs. prepare_fingerprints keeps exactly ONE utterance per session, so this
# degenerates to excluding the diagonal — printed below so that's visible, not assumed.
_excl  = _rid[:, None] == _rid[None, :]
_same  = (_lab[:, None] == _lab[None, :]) & (~_excl)
_valid = ~_excl
_rel        = _same.sum(dim=1)
_n_valid    = _valid.sum(dim=1)
_chance_r1  = float((_rel.float() / _n_valid.clamp(min=1).float()).mean())
_ok_rows    = _rel > 0

print(f"[probe] bank={_N}/{_n_all} clips | {len(set(_pid.tolist()))} speakers | "
      f"{int(_rel.float().mean())} same-speaker targets per query (median "
      f"{int(_rel.median())})")
print(f"[probe] excluded pairs per query: {float(_excl.sum(1).float().mean()):.2f} "
      f"(1.0 == self only, i.e. one utterance per session as expected)")
print(f"[probe] queries with >=1 target: {int(_ok_rows.sum())}/{_N} | "
      f"chance Rank-1 = {_chance_r1:.4f} | chance EER = 0.5")
print()

# ── Feature builders ──────────────────────────────────────────────────────────
def _probe_features(key, l1_norm):
    if key == "random":
        g = torch.Generator(device="cpu").manual_seed(PROBE_SEED)
        X = torch.randn(_N, 512, generator=g).to(device)
    elif key == "all4":
        # Concatenate AFTER per-block L1 normalisation, otherwise the two 5888-d
        # weight blocks outweigh the two 128-d activity blocks ~46:1 by dimension.
        X = torch.cat([_probe_features(k, True) for k in
                       ("in_weights", "hid_weights", "input_activity", "hidden_activity")],
                      dim=1)
        return X
    else:
        a = np.asarray(getattr(PROBE_BANK, key)[_rows]).astype(np.float32).reshape(_N, -1)
        X = torch.from_numpy(np.nan_to_num(a, nan=0.0, posinf=0.0, neginf=0.0)).to(device)
    if l1_norm:
        X = X / X.abs().sum(dim=1, keepdim=True).clamp(min=1e-8)
    return X

# ── Retrieval metrics under L1 distance ───────────────────────────────────────
def _probe_eval(X):
    """Same metric definitions as retrieval_metrics(), scored by -L1 distance.
    EER is computed exactly here (global sort over every valid pair) rather than
    from a 20000-bin histogram — N is small enough that exactness is free."""
    ranks = torch.arange(1, _N + 1, device=device, dtype=torch.float32).view(1, -1)
    r1 = r5 = 0; ap_sum = 0.0; n_ok = 0
    s_parts, y_parts = [], []

    for s in range(0, _N, PROBE_ROW_BLOCK):
        e = X[s:s + PROBE_ROW_BLOCK]
        d = torch.cdist(e, X, p=1)                     # (b, N) L1 distance
        sc = -d                                        # higher = more similar
        blk = slice(s, s + e.shape[0])
        ex, sm, vl = _excl[blk], _same[blk], _valid[blk]
        sc = sc.masked_fill(ex, -float("inf"))
        order = torch.argsort(sc, dim=1, descending=True)
        sm_sorted = torch.gather(sm, 1, order).float()
        r1 += sm_sorted[:, 0].sum().item()
        r5 += (sm_sorted[:, :5].sum(dim=1) > 0).sum().item()
        prec = torch.cumsum(sm_sorted, dim=1) / ranks
        rel  = sm.sum(dim=1)
        ap   = (prec * sm_sorted).sum(dim=1) / rel.clamp(min=1)
        ok   = rel > 0
        ap_sum += float(ap[ok].sum()); n_ok += int(ok.sum())

        s_parts.append(sc[vl]); y_parts.append(sm[vl])

    # Exact EER over all valid pairs (each unordered pair appears twice — that
    # affects neither FRR nor FAR, both being ratios within their own class).
    sc_all = torch.cat(s_parts); y_all = torch.cat(y_parts).float()
    del s_parts, y_parts
    o   = torch.argsort(sc_all, descending=True)
    y   = y_all[o]
    tp  = torch.cumsum(y, 0);        fp = torch.cumsum(1.0 - y, 0)
    frr = 1.0 - tp / tp[-1].clamp(min=1);  far = fp / fp[-1].clamp(min=1)
    k   = int(torch.argmin((frr - far).abs()))
    eer = float((frr[k] + far[k]) / 2)
    del sc_all, y_all, o, y, tp, fp, frr, far
    return dict(eer=eer, rank1=r1 / max(n_ok, 1), rank5=r5 / max(n_ok, 1),
                mAP=ap_sum / max(n_ok, 1))

# ── Run ───────────────────────────────────────────────────────────────────────
print(f"{'component':>16s} {'norm':>5s} {'dim':>6s} | {'EER':>7s} {'R@1':>7s} "
      f"{'R@5':>7s} {'mAP':>7s} | {'sec':>5s}")
print("-" * 74)
_probe_results = {}
for _key, _nrm in PROBE_CONFIGS:
    _t0 = time.time()
    _X = _probe_features(_key, _nrm)
    _m = _probe_eval(_X)
    _probe_results[(_key, _nrm)] = _m
    print(f"{_key:>16s} {'L1' if _nrm else '-':>5s} {_X.shape[1]:>6d} | "
          f"{_m['eer']:>7.4f} {_m['rank1']:>7.4f} {_m['rank5']:>7.4f} "
          f"{_m['mAP']:>7.4f} | {time.time()-_t0:>5.0f}", flush=True)
    del _X
    torch.cuda.empty_cache()

print("-" * 74)
print(f"chance: EER 0.5000, R@1 {_chance_r1:.4f}. ECAPA on this eval set (2s clips, "
      f"full 22496 gallery): EER 0.1328 no-FiLM / 0.1314 FiLM.")
print("NOTE: EER is comparable across gallery sizes; R@1/R@5/mAP on a "
      f"{_N}-clip subsample are optimistic vs the full 22496-clip gallery.")


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  TEMPORARY DIAGNOSTIC #2 — WHY is there so little identity in the fingerprints?
#  Delete once answered. Self-contained; every name is PROBE2_*/_q*.
#
#  The L1 probe said the signal is real but ~useless (EER 0.40) and sits in the
#  INPUT side, not the STDP-learned side. This cell tests the suspected mechanism:
#  that every clip's fingerprint is pinned near one shared attractor, so there is
#  barely any between-clip variance for identity to live in.
#
#  Four questions, per component:
#    1. HOW MUCH does a fingerprint vary between clips at all?  (spread ||sd||/||mean||,
#       and what fraction of entries are exactly 0 from silent-cell masking)
#    2. Is that variance ONE global factor?                     (PCA: var explained by PC1)
#    3. Is that factor just loudness / silence?                 (|corr| of PC1 with total
#       mass and with per-sample zero-fraction)
#    4. Is ANY of the variance speaker-related?                 (Fisher ratio F =
#       between-speaker var / within-speaker var, per dim. F = 1 means no speaker
#       structure whatsoever; the `random` control row shows the empirical null.)
#  Plus, for in_weights only: how far did STDP actually move the weights off the
#  shared W_IH_INIT that every wav starts from?
#
#  PREREQUISITES: the CONFIG cell and the fingerprint-cache cell only (`train_fp`,
#  `test_fp`, `device`). No audio decode, no training. Runs in ~1-2 minutes.
#
#  Sampling: whole speakers (not random clips), so within-speaker variance is
#  estimated from ~25 clips each rather than ~1. Defaults to the vox2 train bank
#  because vox1 averages only ~4 clips/speaker.
# ══════════════════════════════════════════════════════════════════════════════
import numpy as np, torch, time

PROBE2_BANK        = train_fp   # vox2: ~25 clips/speaker -> usable within-speaker variance
PROBE2_N_SPEAKERS  = 300
PROBE2_MAX_PER_SPK = 30
PROBE2_SEED        = 0
PROBE2_KEYS        = ("in_weights", "hid_weights", "input_activity", "hidden_activity")

# ── Sample whole speakers ─────────────────────────────────────────────────────
_q_rng = np.random.default_rng(PROBE2_SEED)
_q_pid_all = PROBE2_BANK.person_ids
_q_speakers = _q_rng.choice(np.unique(_q_pid_all), PROBE2_N_SPEAKERS, replace=False)
_q_rows = []
for _s in _q_speakers:
    _idx = np.flatnonzero(_q_pid_all == _s)
    if len(_idx) > PROBE2_MAX_PER_SPK:
        _idx = _q_rng.choice(_idx, PROBE2_MAX_PER_SPK, replace=False)
    _q_rows.append(_idx)
_q_rows = np.sort(np.concatenate(_q_rows))
_q_N = len(_q_rows)
_q_spk = torch.as_tensor(np.unique(_q_pid_all[_q_rows], return_inverse=True)[1], device=device)
_q_S = int(_q_spk.max()) + 1
print(f"[diag] {_q_N} clips from {_q_S} speakers "
      f"({_q_N/_q_S:.1f} clips/speaker) out of {PROBE2_BANK.n}\n")


def _q_load(key):
    if key == "random":
        g = torch.Generator(device="cpu").manual_seed(PROBE2_SEED)
        return torch.randn(_q_N, 512, generator=g).to(device)
    a = np.asarray(getattr(PROBE2_BANK, key)[_q_rows]).astype(np.float32).reshape(_q_N, -1)
    return torch.from_numpy(np.nan_to_num(a, nan=0.0, posinf=0.0, neginf=0.0)).to(device)


def _q_corr(a, b):
    a = a - a.mean(); b = b - b.mean()
    d = (a.norm() * b.norm()).clamp(min=1e-12)
    return float((a @ b / d).abs())


def _q_analyse(X):
    N, D = X.shape
    mu, sd = X.mean(0), X.std(0)
    spread   = float(sd.norm() / mu.norm().clamp(min=1e-12))
    zero_fr  = float((X == 0).float().mean())
    dead     = float((sd < 1e-6).float().mean())          # entries constant across all clips

    Xc  = X - mu
    tot = (Xc ** 2).sum().clamp(min=1e-12)
    q   = int(min(64, D, N - 1))
    U, Sv, _ = torch.pca_lowrank(Xc, q=q, center=False)
    ev  = (Sv ** 2) / tot
    pc1_scores = U[:, 0] * Sv[0]

    mass = X.abs().sum(1)
    zfr  = (X == 0).float().mean(1)

    # Fisher ratio per dim: between-speaker variance / pooled within-speaker variance.
    # F = 1 under the null (no speaker structure) — see the `random` control row.
    cnt  = torch.zeros(_q_S, device=X.device).index_add_(
        0, _q_spk, torch.ones(N, device=X.device))
    sums = torch.zeros(_q_S, D, device=X.device).index_add_(0, _q_spk, X)
    mu_s = sums / cnt[:, None].clamp(min=1)
    within  = ((X - mu_s[_q_spk]) ** 2).sum(0) / max(N - _q_S, 1)
    between = (cnt[:, None] * (mu_s - mu) ** 2).sum(0) / max(_q_S - 1, 1)
    live = sd > 1e-6
    F = (between / within.clamp(min=1e-12))[live]

    return dict(D=D, spread=spread, zero_fr=zero_fr, dead=dead,
                pc1=float(ev[0]), pc5=float(ev[:5].sum()), pc20=float(ev[:20].sum()),
                c_mass=_q_corr(pc1_scores, mass), c_zero=_q_corr(pc1_scores, zfr),
                F_med=float(F.median()) if F.numel() else float("nan"),
                F_p99=float(torch.quantile(F, 0.99)) if F.numel() else float("nan"))


print(f"{'component':>16s} {'dim':>6s} {'spread':>7s} {'zero%':>6s} {'dead%':>6s} | "
      f"{'PC1':>6s} {'PC1-5':>6s} {'PC1-20':>7s} | {'|r|mass':>7s} {'|r|zero':>7s} | "
      f"{'F med':>6s} {'F p99':>6s}")
print("-" * 108)
_q_res = {}
for _k in PROBE2_KEYS + ("random",):
    _X = _q_load(_k)
    _r = _q_analyse(_X)
    _q_res[_k] = _r
    print(f"{_k:>16s} {_r['D']:>6d} {_r['spread']:>7.3f} {_r['zero_fr']*100:>5.1f}% "
          f"{_r['dead']*100:>5.1f}% | {_r['pc1']:>6.3f} {_r['pc5']:>6.3f} {_r['pc20']:>7.3f} | "
          f"{_r['c_mass']:>7.3f} {_r['c_zero']:>7.3f} | {_r['F_med']:>6.2f} {_r['F_p99']:>6.2f}",
          flush=True)
    del _X; torch.cuda.empty_cache()
print("-" * 108)
print("spread = ||sd||/||mean|| across clips (small => every clip's fingerprint is nearly")
print("         the same vector). zero% = entries zeroed by silent-cell masking.")
print("dead%  = entries identical in EVERY clip. PC1..  = fraction of between-clip variance")
print("         explained. |r|mass/|r|zero = how much PC1 is just loudness / silence.")
print("F      = between-speaker / within-speaker variance per dim; 1.0 = no speaker structure")
print("         (check the `random` row: it should sit at ~1.0).\n")

# ── in_weights only: how far did STDP move off the shared init? ───────────────
# W_IH_INIT is identical for every wav, so if the corpus mean fingerprint is still
# ~= init and per-clip displacement is small, 2s of exposure never left the attractor.
# Formulas verbatim from prepare_fingerprints.ipynb's _fingerprint_core.py.
_qN_IN, _qN_H, _qN_CH, _qN_PC, _qR, _qP, _qNORM = 128, 128, 64, 2, 11, 3, 1.0455
_qd = np.abs((np.arange(_qN_IN) // _qN_PC).reshape(-1, 1) - (np.arange(_qN_H) // _qN_PC).reshape(1, -1))
_qd = np.minimum(_qd, _qN_CH - _qd)
_qtopo = np.maximum(0.0, 1.0 - (_qd / _qR) ** _qP)
_qsrc, _qtgt = np.where(_qd <= _qR)
_qW = np.zeros((_qN_IN, _qN_H)); _qW[_qsrc, _qtgt] = _qtopo[_qsrc, _qtgt]
for _j in range(_qN_H):
    _qr = _qsrc[_qtgt == _j]; _qs = _qW[_qr, _j].sum()
    if _qs > 0: _qW[_qr, _j] *= _qNORM / _qs
_qoff = (np.arange(_qN_H) // _qN_PC)[None, :] + np.arange(-_qR, _qR + 1)[:, None]
_qIDX = np.stack([(_qoff % _qN_CH) * _qN_PC + t for t in range(_qN_PC)])
_qINIT = torch.from_numpy(
    _qW[_qIDX, np.broadcast_to(np.arange(_qN_H), _qIDX.shape)].astype(np.float32)
).reshape(-1).to(device)

_X = _q_load("in_weights")
_qlive = (_X != 0) & (_qINIT[None, :] != 0)     # ignore silent-cell zeros, which are a
                                                 # masking artefact, not an STDP change
_qrel = ((_X - _qINIT[None, :]).abs() * _qlive).sum(1) / (_qINIT[None, :].abs() * _qlive).sum(1).clamp(min=1e-12)
_qmean = _X.mean(0)
print(f"[init] per-clip relative displacement from W_IH_INIT (non-masked entries only):")
print(f"       median {float(_qrel.median()):.4f}  p10 {float(torch.quantile(_qrel,0.1)):.4f}"
      f"  p90 {float(torch.quantile(_qrel,0.9)):.4f}")
print(f"[init] corpus-mean fingerprint vs init: cosine "
      f"{float(torch.nn.functional.cosine_similarity(_qmean, _qINIT, dim=0)):.4f}")
print(f"[init] between-clip spread vs displacement-from-init: "
      f"{_q_res['in_weights']['spread']:.4f} vs {float(_qrel.median()):.4f}")
print("       If displacement is LARGE but spread is SMALL, STDP moves every clip to the")
print("       SAME new place (a shared attractor). If BOTH are small, STDP barely moved")
print("       anything in 2s. The two imply different upstream fixes.")
del _X; torch.cuda.empty_cache()


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  TEMPORARY DIAGNOSTIC #3 — how much speaker identity is EXTRACTABLE?
#  Delete once answered. Self-contained; every name is PROBE3_*/_r3*.
#
#  Diagnostic #2 found real speaker structure (F 3.0-5.7 vs null 0.99) that is
#  low-rank and dominated by an energy/silence factor (PC1 |r| 0.90 with total
#  mass). That explains why raw L1 retrieval looked near-chance without proving
#  the fingerprints are empty. Two tests here, cheap -> expensive:
#
#  PART A (unsupervised, seconds): re-run the SAME retrieval after removing the
#    energy directions. If EER moves sharply off 0.40, energy-dominance is
#    confirmed as the reason the first probe failed.
#  PART B (supervised, minutes): train a linear head and an MLP on vox2
#    fingerprints -> speaker ID, then evaluate retrieval on HELD-OUT vox1
#    speakers. This upper-bounds what FiLM could ever extract from these
#    fingerprints, and is the honest decider for the whole direction.
#
#  All PCA / whitening / standardisation statistics are fit on vox2 (train) and
#  applied to vox1 (eval) — never fit on the evaluation set.
#
#  PREREQUISITES: the CONFIG cell and the fingerprint-cache cell only
#  (`train_fp`, `test_fp`, `device`). No audio decode, no training cells.
#
#  The eval subsample is byte-identical to diagnostic #1's (same seed, same draw),
#  so the "raw / L1" rows below MUST reproduce that table — a built-in check.
# ══════════════════════════════════════════════════════════════════════════════
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F, time

PROBE3_KEYS      = ("in_weights", "hid_weights", "input_activity", "hidden_activity")
PROBE3_EVAL_N    = 5000     # vox1 clips; matches diagnostic #1 exactly
PROBE3_FIT_N     = 10000    # vox2 clips used to fit PCA / standardisation
PROBE3_TRAIN_N   = 80000    # vox2 clips used to train the supervised probe
PROBE3_SEED      = 0
PROBE3_EPOCHS    = 20
PROBE3_BATCH     = 1024
PROBE3_EMBED     = 192      # same embedding width ECAPA uses
PROBE3_SCALE     = 30       # cosine-head scale (AAMSoftmax with m=0)

_r3_dev = device

# ── Eval-set masks (identical protocol + identical rows as diagnostic #1) ─────
_r3_rows = np.sort(np.random.default_rng(PROBE3_SEED).choice(
    test_fp.n, PROBE3_EVAL_N, replace=False))
_r3_pid = test_fp.person_ids[_r3_rows]
_r3_sid = test_fp.session_ids[_r3_rows]
_r3_lab = torch.as_tensor(np.unique(_r3_pid, return_inverse=True)[1], device=_r3_dev)
_r3_rec = torch.as_tensor(np.unique(
    np.char.add(np.char.add(_r3_pid, "/"), _r3_sid), return_inverse=True)[1], device=_r3_dev)
_r3_excl  = _r3_rec[:, None] == _r3_rec[None, :]
_r3_same  = (_r3_lab[:, None] == _r3_lab[None, :]) & (~_r3_excl)
_r3_valid = ~_r3_excl
_r3_N     = len(_r3_rows)


def _r3_read(bank, rows, keys=PROBE3_KEYS, dtype=np.float32):
    """Concatenated feature matrix for `rows` of `bank`, allocated once at `dtype`
    and filled per component. Building it with concatenate+astype+nan_to_num instead
    would peak at ~4x this size on the 80k-row training read."""
    n = len(rows)
    dims = [int(np.prod(getattr(bank, k).shape[1:])) for k in keys]
    out = np.empty((n, sum(dims)), dtype=dtype)
    o = 0
    for k, d in zip(keys, dims):
        out[:, o:o + d] = np.asarray(getattr(bank, k)[rows]).reshape(n, -1)
        o += d
    np.nan_to_num(out, copy=False, nan=0.0, posinf=0.0, neginf=0.0)
    return torch.from_numpy(out)


def _r3_eval(X, metric="l1", block=256):
    """Session-free all-pairs retrieval on the eval subsample. Same metric
    definitions as retrieval_metrics(); EER exact via a global sort."""
    X = X.to(_r3_dev)
    if metric == "cos":
        X = F.normalize(X, dim=1)
    ranks = torch.arange(1, _r3_N + 1, device=_r3_dev, dtype=torch.float32).view(1, -1)
    r1 = r5 = 0; ap_sum = 0.0; n_ok = 0; s_parts, y_parts = [], []
    for s in range(0, _r3_N, block):
        e = X[s:s + block]
        sc = (e @ X.t()) if metric == "cos" else (-torch.cdist(e, X, p=1))
        blk = slice(s, s + e.shape[0])
        ex, sm, vl = _r3_excl[blk], _r3_same[blk], _r3_valid[blk]
        sc = sc.masked_fill(ex, -float("inf"))
        order = torch.argsort(sc, dim=1, descending=True)
        sm_s = torch.gather(sm, 1, order).float()
        r1 += sm_s[:, 0].sum().item()
        r5 += (sm_s[:, :5].sum(dim=1) > 0).sum().item()
        rel = sm.sum(dim=1); ok = rel > 0
        ap = ((torch.cumsum(sm_s, dim=1) / ranks) * sm_s).sum(dim=1) / rel.clamp(min=1)
        ap_sum += float(ap[ok].sum()); n_ok += int(ok.sum())
        s_parts.append(sc[vl]); y_parts.append(sm[vl])
    sc_all = torch.cat(s_parts); y_all = torch.cat(y_parts).float()
    del s_parts, y_parts
    o = torch.argsort(sc_all, descending=True); y = y_all[o]
    tp = torch.cumsum(y, 0); fp = torch.cumsum(1.0 - y, 0)
    frr = 1.0 - tp / tp[-1].clamp(min=1); far = fp / fp[-1].clamp(min=1)
    k = int(torch.argmin((frr - far).abs()))
    out = dict(eer=float((frr[k] + far[k]) / 2), rank1=r1 / max(n_ok, 1),
               rank5=r5 / max(n_ok, 1), mAP=ap_sum / max(n_ok, 1))
    del sc_all, y_all, o, y, tp, fp, frr, far, X
    torch.cuda.empty_cache()
    return out


def _r3_row(name, variant, metric, D, m, t0):
    print(f"{name:>16s} {variant:>9s} {metric:>4s} {D:>6d} | {m['eer']:>7.4f} "
          f"{m['rank1']:>7.4f} {m['rank5']:>7.4f} {m['mAP']:>7.4f} | {time.time()-t0:>5.0f}",
          flush=True)


# ══════════════════════════════════════════════════════════════════════════════
#  PART A — does removing the energy directions rescue unsupervised retrieval?
# ══════════════════════════════════════════════════════════════════════════════
_r3_fit_rows = np.sort(np.random.default_rng(PROBE3_SEED + 1).choice(
    train_fp.n, PROBE3_FIT_N, replace=False))

print("PART A — unsupervised retrieval, energy directions removed "
      "(PCA fit on vox2, applied to vox1)\n")
print(f"{'component':>16s} {'variant':>9s} {'dist':>4s} {'dim':>6s} | {'EER':>7s} "
      f"{'R@1':>7s} {'R@5':>7s} {'mAP':>7s} | {'sec':>5s}")
print("-" * 82)

for _k in PROBE3_KEYS:
    _Xf = _r3_read(train_fp, _r3_fit_rows, (_k,)).to(_r3_dev)
    _mu = _Xf.mean(0)
    _q  = int(min(64, _Xf.shape[1], PROBE3_FIT_N - 1))
    _U, _S, _V = torch.pca_lowrank(_Xf - _mu, q=_q, center=False)
    _sd_pc = (_S / np.sqrt(PROBE3_FIT_N - 1)).clamp(min=1e-6)
    del _Xf, _U; torch.cuda.empty_cache()

    _Xe = _r3_read(test_fp, _r3_rows, (_k,)).to(_r3_dev)
    _Xc = _Xe - _mu
    _variants = {
        "raw":      _Xe,
        "-PC1":     _Xc - (_Xc @ _V[:, :1]) @ _V[:, :1].t(),
        "-PC5":     _Xc - (_Xc @ _V[:, :5]) @ _V[:, :5].t(),
        "whiten64": (_Xc @ _V) / _sd_pc,
    }
    for _vn, _Xv in _variants.items():
        for _met in ("l1", "cos"):
            _t0 = time.time()
            _r3_row(_k, _vn, _met, _Xv.shape[1], _r3_eval(_Xv, _met), _t0)
    del _Xe, _Xc, _variants, _V; torch.cuda.empty_cache()
print("-" * 82)
print("`raw`/`l1` rows must match diagnostic #1's table exactly (same rows, same protocol).")
print("chance EER 0.5 | ECAPA on this eval set: 0.1328 no-FiLM / 0.1314 FiLM.\n")


# ══════════════════════════════════════════════════════════════════════════════
#  PART B — supervised probe: upper bound on extractable identity
# ══════════════════════════════════════════════════════════════════════════════
print("PART B — supervised probe (train on vox2 speakers, eval on held-out vox1)\n")

_r3_tr_rows = np.sort(np.random.default_rng(PROBE3_SEED + 2).choice(
    train_fp.n, min(PROBE3_TRAIN_N, train_fp.n), replace=False))
_r3_Xtr = _r3_read(train_fp, _r3_tr_rows, dtype=np.float16)   # keep in RAM as f16
_r3_ytr = torch.from_numpy(
    np.unique(train_fp.person_ids[_r3_tr_rows], return_inverse=True)[1]).long()
_r3_C = int(_r3_ytr.max()) + 1
_r3_D = _r3_Xtr.shape[1]
print(f"[probe] train {len(_r3_tr_rows)} clips / {_r3_C} speakers "
      f"({len(_r3_tr_rows)/_r3_C:.1f} per speaker) | D={_r3_D}")

# Standardise with vox2 statistics only.
_r3_fit = _r3_read(train_fp, _r3_fit_rows).to(_r3_dev)
_r3_mu, _r3_sd = _r3_fit.mean(0), _r3_fit.std(0).clamp(min=1e-4)
del _r3_fit; torch.cuda.empty_cache()

_r3_Xev = ((_r3_read(test_fp, _r3_rows).to(_r3_dev) - _r3_mu) / _r3_sd)


class _R3Head(nn.Module):
    """Linear or 1-hidden-layer MLP -> L2-normalised embedding + cosine classifier
    (AAMSoftmax with m=0; the margin is irrelevant for measuring extractability)."""
    def __init__(self, D, C, embed, hidden=None):
        super().__init__()
        self.body = nn.Linear(D, embed) if hidden is None else nn.Sequential(
            nn.Linear(D, hidden), nn.BatchNorm1d(hidden), nn.ReLU(), nn.Linear(hidden, embed))
        self.W = nn.Parameter(torch.empty(C, embed)); nn.init.xavier_normal_(self.W)

    def embed(self, x):
        return self.body(x)

    def forward(self, x, y):
        cos = F.linear(F.normalize(self.embed(x)), F.normalize(self.W))
        return F.cross_entropy(cos * PROBE3_SCALE, y)


def _r3_train_probe(hidden, tag):
    torch.manual_seed(PROBE3_SEED)
    net = _R3Head(_r3_D, _r3_C, PROBE3_EMBED, hidden).to(_r3_dev)
    opt = torch.optim.Adam(net.parameters(), lr=1e-3, weight_decay=1e-4)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=PROBE3_EPOCHS)
    n = len(_r3_tr_rows); rng = np.random.default_rng(PROBE3_SEED)
    t0 = time.time()
    for ep in range(PROBE3_EPOCHS):
        net.train(); perm = rng.permutation(n); tot = 0.0; hit = 0; seen = 0
        for s in range(0, n - PROBE3_BATCH + 1, PROBE3_BATCH):
            idx = perm[s:s + PROBE3_BATCH]
            xb = ((_r3_Xtr[idx].to(_r3_dev).float() - _r3_mu) / _r3_sd)
            yb = _r3_ytr[idx].to(_r3_dev)
            loss = net(xb, yb)
            opt.zero_grad(set_to_none=True); loss.backward(); opt.step()
            with torch.no_grad():
                cos = F.linear(F.normalize(net.embed(xb)), F.normalize(net.W))
                hit += int((cos.argmax(1) == yb).sum()); seen += len(idx)
            tot += float(loss) * len(idx)
        sch.step()
        if ep % 5 == 4 or ep == PROBE3_EPOCHS - 1:
            print(f"  [{tag} e{ep:02d}] loss={tot/seen:.4f} train_acc={hit/seen:.4f} "
                  f"({time.time()-t0:.0f}s)", flush=True)
    net.eval()
    with torch.no_grad():
        emb = torch.cat([net.embed(_r3_Xev[i:i + 2048]) for i in range(0, _r3_N, 2048)])
    return _r3_eval(emb.float(), "cos")


print(f"\n{'model':>16s} {'variant':>9s} {'dist':>4s} {'dim':>6s} | {'EER':>7s} "
      f"{'R@1':>7s} {'R@5':>7s} {'mAP':>7s} | {'sec':>5s}")
print("-" * 82)
_t0 = time.time(); _m = _r3_eval(_r3_Xev, "cos")
_r3_row("no-probe (z)", "identity", "cos", _r3_D, _m, _t0)
for _hid, _tag in ((None, "linear"), (512, "mlp512")):
    print()
    _t0 = time.time(); _m = _r3_train_probe(_hid, _tag)
    _r3_row(_tag, "trained", "cos", PROBE3_EMBED, _m, _t0)
print("-" * 82)
print("`no-probe (z)` is the untrained floor: standardised features, cosine, no learning.")
print("The trained rows are the UPPER BOUND on what FiLM could extract from these")
print("fingerprints. Compare against ECAPA's own 0.1328 (no-FiLM) on this eval set:")
print("if the probe lands near chance, the ~1% FiLM delta was regularisation, not signal.")


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  TEMPORARY DIAGNOSTIC #4 — is the fingerprint COMPLEMENTARY to the audio?
#  Delete once answered. Self-contained; every name is PROBE4_*/_r4*.
#
#  Diagnostic #3 measured how STRONG the fingerprint is on its own (EER 0.345 vs
#  ECAPA's 0.133). That does not answer whether it knows anything the audio does
#  NOT. A weak-but-orthogonal side channel can still improve a strong model; a
#  weak-and-redundant one cannot. This cell decides that, with no retraining of
#  ECAPA: it scores the same vox1 pairs with the saved ECAPA checkpoint and with
#  the fingerprint probe, then fuses the two score matrices.
#
#    fused = cos_ecapa + w * cos_probe,  swept over w
#
#  DELIBERATELY GENEROUS: w is swept on the evaluation set itself, so the best
#  fused row is an optimistic, tuned-on-test number. That is the point — if even
#  the best possible weighting cannot beat w=0 (ECAPA alone), the answer is a
#  clean NO and the direction is dead. If some w does help, the effect is real
#  but would still need honest validation on a held-out split before being claimed.
#
#  PREREQUISITES — run these cells first, in order, then this one:
#    1 env check | 2 CONFIG | 3 fingerprint cache | 4 model definitions |
#    14 diagnostic #3 (supplies the probe data + the shared eval protocol)
#  SKIP the decode/bank/train cells (7-11). This cell decodes ONLY the 5000
#  evaluation clips it needs (~3 min), so the ~90 min corpus decode is not required.
# ══════════════════════════════════════════════════════════════════════════════
import os, glob, subprocess, time
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
from concurrent.futures import ThreadPoolExecutor

PROBE4_CKPT    = os.path.join(CKPT_DIR, "best_ecapa_C1024_nofilm.pt")
PROBE4_WEIGHTS = (0.0, 0.05, 0.1, 0.2, 0.3, 0.5, 0.75, 1.0, 1.5, 2.0)
PROBE4_BATCH   = 64

for _need in ("_r3_rows", "_r3_excl", "_r3_same", "_r3_valid", "_r3_Xev",
              "_r3_Xtr", "_r3_ytr", "_r3_mu", "_r3_sd", "_R3Head"):
    assert _need in dir(), f"run diagnostic #3 (cell 14) first — `{_need}` is undefined"
assert "ECAPA_TDNN" in dir(), "run the model-definition cell (cell 4) first"
assert os.path.exists(PROBE4_CKPT), (
    f"checkpoint not found: {PROBE4_CKPT}\n"
    f"    available in {CKPT_DIR}: {sorted(os.listdir(CKPT_DIR))}\n"
    f"    set PROBE4_CKPT to the no-FiLM best checkpoint (re-upload it if the "
    f"Kaggle session that produced it has ended).")

_r4_N = len(_r3_rows)

# ── 1. Resolve + decode ONLY the 5000 evaluation clips ────────────────────────
def _r4_person_index(pattern, dev_nn_style):
    index = {}
    for root in sorted(glob.glob(pattern)):
        bases = sorted(glob.glob(os.path.join(root, "dev_*"))) if dev_nn_style else [root]
        for base in bases:
            try:
                for p in os.listdir(base):
                    index.setdefault(p, base)
            except OSError:
                pass
    return index

_r4_idx = _r4_person_index(TEST_GLOB, dev_nn_style=True)
_r4_paths = []
for _i in _r3_rows:
    _p, _s, _f = (test_fp.person_ids[_i], test_fp.session_ids[_i], test_fp.file_names[_i])
    _b = _r4_idx.get(_p)
    _r4_paths.append(None if _b is None else os.path.join(_b, _p, _s, _f))
assert all(p is not None for p in _r4_paths), "some eval speakers have no audio directory"

_r4_cap = int(CLIP_SEC * SR)

def _r4_decode(path):
    """First CLIP_SEC of one clip as int16, or None. Same ffmpeg invocation the
    notebook's own decode cache uses (-t stops the decode early)."""
    try:
        pr = subprocess.run(
            ["ffmpeg", "-nostdin", "-loglevel", "error", "-i", path,
             "-t", f"{CLIP_SEC:.3f}", "-ac", "1", "-ar", str(SR),
             "-f", "f32le", "-acodec", "pcm_f32le", "-"], capture_output=True)
        if pr.returncode != 0:
            return None
        y = np.frombuffer(pr.stdout, dtype="<f4").astype(np.float32)[:_r4_cap]
        if y.size == 0:
            return None
        if y.size < _r4_cap:                       # wrap-pad short clips, as WaveBank does
            y = np.pad(y, (0, _r4_cap - y.size), mode="wrap")
        return np.clip(y * 32768.0, -32768, 32767).astype(np.int16)
    except Exception:
        return None

print(f"[fuse] decoding {_r4_N} eval clips (first {CLIP_SEC}s only) ...", flush=True)
_t0 = time.time()
_r4_wav = np.zeros((_r4_N, _r4_cap), dtype=np.int16)
_r4_ok = np.zeros(_r4_N, dtype=bool)
with ThreadPoolExecutor(max_workers=(os.cpu_count() or 2) * 3) as _pool:
    for _k, _a in enumerate(_pool.map(_r4_decode, _r4_paths)):
        if _a is not None:
            _r4_wav[_k] = _a; _r4_ok[_k] = True
print(f"[fuse] decoded {int(_r4_ok.sum())}/{_r4_N} in {time.time()-_t0:.0f}s")
assert _r4_ok.mean() > 0.98, "too many decode failures to trust the comparison"

# ── 2. ECAPA embeddings from the saved checkpoint ─────────────────────────────
_r4_ck = torch.load(PROBE4_CKPT, map_location=device)
print(f"[fuse] checkpoint tag={_r4_ck.get('tag')} epoch={_r4_ck.get('epoch')} "
      f"metrics={_r4_ck.get('metrics')}")
_r4_net = ECAPA_TDNN(C, use_film=False).to(device)
_r4_net.load_state_dict(_r4_ck["model"])
_r4_net.eval()

with torch.no_grad():
    _r4_e = []
    for _s in range(0, _r4_N, PROBE4_BATCH):
        _x = torch.from_numpy(
            _r4_wav[_s:_s + PROBE4_BATCH].astype(np.float32) / 32768.0).to(device)
        with torch.autocast("cuda", dtype=torch.float16):
            _r4_e.append(_r4_net(_x).float())
    _r4_emb_audio = F.normalize(torch.cat(_r4_e), dim=1)
del _r4_net, _r4_e, _r4_ck
torch.cuda.empty_cache()

# ── 3. Fingerprint-probe embeddings (linear head — it transferred best in #3) ──
def _r4_fit_probe(hidden=None, epochs=PROBE3_EPOCHS):
    torch.manual_seed(PROBE3_SEED)
    net = _R3Head(_r3_Xtr.shape[1], int(_r3_ytr.max()) + 1, PROBE3_EMBED, hidden).to(device)
    opt = torch.optim.Adam(net.parameters(), lr=1e-3, weight_decay=1e-4)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    n = _r3_Xtr.shape[0]; rng = np.random.default_rng(PROBE3_SEED)
    for _ in range(epochs):
        net.train()
        perm = rng.permutation(n)
        for s in range(0, n - PROBE3_BATCH + 1, PROBE3_BATCH):
            idx = perm[s:s + PROBE3_BATCH]
            xb = (_r3_Xtr[idx].to(device).float() - _r3_mu) / _r3_sd
            loss = net(xb, _r3_ytr[idx].to(device))
            opt.zero_grad(set_to_none=True); loss.backward(); opt.step()
        sch.step()
    net.eval()
    with torch.no_grad():
        emb = torch.cat([net.embed(_r3_Xev[i:i + 2048]) for i in range(0, _r4_N, 2048)])
    return F.normalize(emb.float(), dim=1)

print("[fuse] training the fingerprint probe ...", flush=True)
_t0 = time.time()
_r4_emb_fp = _r4_fit_probe()
print(f"[fuse] probe trained in {time.time()-_t0:.0f}s")

# ── 4. Score matrices, redundancy, fusion sweep ───────────────────────────────
_r4_S_audio = _r4_emb_audio @ _r4_emb_audio.t()
_r4_S_fp    = _r4_emb_fp    @ _r4_emb_fp.t()

_va = _r4_S_audio[_r3_valid]; _vf = _r4_S_fp[_r3_valid]
_r4_corr = float(((_va - _va.mean()) @ (_vf - _vf.mean())) /
                 ((_va - _va.mean()).norm() * (_vf - _vf.mean()).norm()).clamp(min=1e-12))
print(f"[fuse] correlation between the two systems' pair scores: {_r4_corr:+.4f}")
print("       (near 0 = they judge pairs differently -> room to help each other;")
print("        near 1 = they are measuring the same thing -> nothing to add)")
del _va, _vf


def _r4_eval_scores(S, block=512):
    ranks = torch.arange(1, _r4_N + 1, device=device, dtype=torch.float32).view(1, -1)
    r1 = r5 = 0; ap_sum = 0.0; n_ok = 0; s_parts, y_parts = [], []
    for s in range(0, _r4_N, block):
        sc = S[s:s + block].masked_fill(_r3_excl[s:s + block], -float("inf"))
        sm, vl = _r3_same[s:s + block], _r3_valid[s:s + block]
        order = torch.argsort(sc, dim=1, descending=True)
        sm_s = torch.gather(sm, 1, order).float()
        r1 += sm_s[:, 0].sum().item()
        r5 += (sm_s[:, :5].sum(dim=1) > 0).sum().item()
        rel = sm.sum(dim=1); ok = rel > 0
        ap = ((torch.cumsum(sm_s, dim=1) / ranks) * sm_s).sum(dim=1) / rel.clamp(min=1)
        ap_sum += float(ap[ok].sum()); n_ok += int(ok.sum())
        s_parts.append(sc[vl]); y_parts.append(sm[vl])
    sc_all = torch.cat(s_parts); y_all = torch.cat(y_parts).float()
    o = torch.argsort(sc_all, descending=True); y = y_all[o]
    tp = torch.cumsum(y, 0); fp = torch.cumsum(1.0 - y, 0)
    frr = 1.0 - tp / tp[-1].clamp(min=1); far = fp / fp[-1].clamp(min=1)
    k = int(torch.argmin((frr - far).abs()))
    out = dict(eer=float((frr[k] + far[k]) / 2), rank1=r1 / max(n_ok, 1),
               rank5=r5 / max(n_ok, 1), mAP=ap_sum / max(n_ok, 1))
    del sc_all, y_all, o, y, tp, fp, frr, far
    torch.cuda.empty_cache()
    return out


print(f"\n{'system':>22s} | {'EER':>7s} {'R@1':>7s} {'R@5':>7s} {'mAP':>7s}")
print("-" * 60)
_m = _r4_eval_scores(_r4_S_fp)
print(f"{'fingerprint alone':>22s} | {_m['eer']:>7.4f} {_m['rank1']:>7.4f} "
      f"{_m['rank5']:>7.4f} {_m['mAP']:>7.4f}")
print("-" * 60)
_r4_base = None
for _w in PROBE4_WEIGHTS:
    _m = _r4_eval_scores(_r4_S_audio + _w * _r4_S_fp)
    if _w == 0.0:
        _r4_base = _m["eer"]
        _tag = "ECAPA alone (w=0.00)"
    else:
        _d = _r4_base - _m["eer"]
        _tag = f"+ fingerprint w={_w:<5.2f}"
    _extra = "" if _w == 0.0 else f"   {'better' if _d > 0 else 'worse '} by {abs(_d):.4f}"
    print(f"{_tag:>22s} | {_m['eer']:>7.4f} {_m['rank1']:>7.4f} {_m['rank5']:>7.4f} "
          f"{_m['mAP']:>7.4f}{_extra}")
print("-" * 60)
print(f"Reference: this ECAPA scored EER 0.1328 on the full 22496-clip gallery; the")
print(f"w=0.00 row above is the same model on this {_r4_N}-clip subsample and is THE")
print("baseline to compare against — every other row is measured on identical pairs.")
print("If no w beats w=0.00, the fingerprint adds nothing the audio did not already")
print("have, and that conclusion is solid because w was tuned on the test set itself.")
